In [20]:
import pandas as pd

csv_path = 'C:\\Users\\athar\\smartgrowth-ai\\data\\WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(csv_path)

df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [40]:
df_customers = df[['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 
                       'tenure', 'Contract', 'MonthlyCharges', 'TotalCharges', 'Churn']].copy()
    
    # Rename columns to match SQL schema
df_customers.columns = ['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents', 
                            'tenure_months', 'subscription_type', 'monthly_charges', 'total_charges', 'churn_status']
    
    # Convert data types
df_customers['senior_citizen'] = df_customers['senior_citizen'].astype(bool)
df_customers['partner'] = df_customers['partner'].map({'Yes': True, 'No': False})
df_customers['dependents'] = df_customers['dependents'].map({'Yes': True, 'No': False})  
df_customers['churn_status'] = df_customers['churn_status'].map({'Yes': True, 'No': False})
    
    # Handle TotalCharges column (some entries are spaces, need to convert to numeric)
df_customers['total_charges'] = pd.to_numeric(df_customers['total_charges'], errors='coerce')
    
    # Fill missing total_charges with calculated value
missing_total = df_customers['total_charges'].isna().sum()
if missing_total > 0:
        # logger.info(f"Found {missing_total} missing total_charges values - filling with calculated values")
    df_customers['total_charges'] = df_customers['total_charges'].fillna(
    df_customers['monthly_charges'] * df_customers['tenure_months']
    )
    
df_customers
df = df_customers
df

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,subscription_type,monthly_charges,total_charges,churn_status
0,7590-VHVEG,Female,False,True,False,1,Month-to-month,29.85,29.85,False
1,5575-GNVDE,Male,False,False,False,34,One year,56.95,1889.50,False
2,3668-QPYBK,Male,False,False,False,2,Month-to-month,53.85,108.15,True
3,7795-CFOCW,Male,False,False,False,45,One year,42.30,1840.75,False
4,9237-HQITU,Female,False,False,False,2,Month-to-month,70.70,151.65,True
...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,False,True,True,24,One year,84.80,1990.50,False
7039,2234-XADUH,Female,False,True,True,72,One year,103.20,7362.90,False
7040,4801-JZAZL,Female,False,True,True,11,Month-to-month,29.60,346.45,False
7041,8361-LTMKD,Male,True,True,False,4,Month-to-month,74.40,306.60,True


In [48]:
print(f"\n📊 Dataset Overview:")
print(f"  - Total customers: {len(df):,}")
print(f"  - Features: {len(df.columns)}")
print(f"  - Memory usage: {df.memory_usage().sum() / 1024:.1f} KB")

# 2. Target variable analysis
churn_rate = df['churn_status'].mean()
print(f"\n🎯 Target Variable (Churn):")
print(f"  - Churn rate: {churn_rate:.1%}")
print(f"  - Churned customers: {df['churn_status'].sum():,}")
print(f"  - Retained customers: {(~df['churn_status']).sum():,}")

# # 3. Customer segments
# print(f"\n👥 Customer Segments:")
# segments = pd.read_sql("SELECT customer_segment, COUNT(*) as count FROM customer_summary GROUP BY customer_segment ORDER BY count DESC", conn)
# for _, row in segments.iterrows():
#     print(f"  - {row['customer_segment']}: {row['count']:,}")

# # 4. Spending analysis
# print(f"\n💰 Spending Analysis:")
# spending = pd.read_sql("SELECT spending_tier, COUNT(*) as count, AVG(monthly_charges) as avg_monthly FROM customer_summary GROUP BY spending_tier ORDER BY avg_monthly", conn)
# for _, row in spending.iterrows():
#     print(f"  - {row['spending_tier']} spenders: {row['count']:,} customers (avg: ${row['avg_monthly']:.2f}/month)")

# 5. Tenure analysis
print(f"\n⏰ Customer Tenure:")
print(f"  - Average tenure: {df['tenure_months'].mean():.1f} months")
print(f"  - Median tenure: {df['tenure_months'].median():.1f} months")
print(f"  - New customers (< 12 months): {(df['tenure_months'] < 12).sum():,}")
print(f"  - Long-term customers (> 36 months): {(df['tenure_months'] > 36).sum():,}")

# 6. Contract type analysis
print(f"\n📋 Contract Types:")
contracts = df['subscription_type'].value_counts()
for contract_type, count in contracts.items():
    pct = count / len(df) * 100
    print(f"  - {contract_type}: {count:,} ({pct:.1f}%)")

# 7. Demographics
print(f"\n👤 Demographics:")
print(f"  - Female customers: {(df['gender'] == 'Female').sum():,} ({(df['gender'] == 'Female').mean():.1%})")
print(f"  - Male customers: {(df['gender'] == 'Male').sum():,} ({(df['gender'] == 'Male').mean():.1%})")
print(f"  - Senior citizens: {df['senior_citizen'].sum():,} ({df['senior_citizen'].mean():.1%})")
print(f"  - Customers with partners: {df['partner'].sum():,} ({df['partner'].mean():.1%})")
print(f"  - Customers with dependents: {df['dependents'].sum():,} ({df['dependents'].mean():.1%})")

# 8. Key insights for ML
print(f"\n🤖 Key Insights for ML Models:")

# Churn by contract type
churn_by_contract = df.groupby('subscription_type')['churn_status'].mean().sort_values(ascending=False)
print(f"  - Highest churn rate: {churn_by_contract.index[0]} ({churn_by_contract.iloc[0]:.1%})")
print(f"  - Lowest churn rate: {churn_by_contract.index[-1]} ({churn_by_contract.iloc[-1]:.1%})")

# Churn by tenure
short_tenure_churn = df[df['tenure_months'] < 12]['churn_status'].mean()
long_tenure_churn = df[df['tenure_months'] > 36]['churn_status'].mean()
print(f"  - New customer churn rate: {short_tenure_churn:.1%}")
print(f"  - Long-term customer churn rate: {long_tenure_churn:.1%}")

# Revenue impact - Fixed boolean indexing
churned_customers = df[df['churn_status'] == True]  # Explicit boolean comparison
churned_revenue = churned_customers['monthly_charges'].sum()
total_revenue = df['monthly_charges'].sum()
print(f"  - Revenue at risk from churn: ${churned_revenue:,.0f}/month ({churned_revenue/total_revenue:.1%})")
 


📊 Dataset Overview:
  - Total customers: 7,043
  - Features: 10
  - Memory usage: 357.8 KB

🎯 Target Variable (Churn):
  - Churn rate: 26.5%
  - Churned customers: 1,869
  - Retained customers: 5,174

⏰ Customer Tenure:
  - Average tenure: 32.4 months
  - Median tenure: 29.0 months
  - New customers (< 12 months): 2,069
  - Long-term customers (> 36 months): 3,001

📋 Contract Types:
  - Month-to-month: 3,875 (55.0%)
  - Two year: 1,695 (24.1%)
  - One year: 1,473 (20.9%)

👤 Demographics:
  - Female customers: 3,488 (49.5%)
  - Male customers: 3,555 (50.5%)
  - Senior citizens: 1,142 (16.2%)
  - Customers with partners: 3,402 (48.3%)
  - Customers with dependents: 2,110 (30.0%)

🤖 Key Insights for ML Models:
  - Highest churn rate: Month-to-month (42.7%)
  - Lowest churn rate: Two year (2.8%)
  - New customer churn rate: 48.3%
  - Long-term customer churn rate: 11.9%
  - Revenue at risk from churn: $139,131/month (30.5%)
